# Spotify + Cassandra: Full Starter Notebook

This notebook reads `Spotify_Youtube.csv`, cleans it, creates processed song and artist-summary outputs, and writes those outputs to Cassandra. It does **not** copy the entire raw CSV into Cassandra.

## Before running
- Start Cassandra and keep it running.
- Install dependencies in the same Python environment as Jupyter: `py -3.11 -m pip install pandas cassandra-driver jupyter`
- Put `Spotify_Youtube.csv` next to this notebook, or edit `CSV_PATH`.
- Run the cells from top to bottom.

This is a starter integration because your exact existing project notebook/code has not been attached in this turn. Column names are detected from common Spotify/YouTube CSV variants; adjust the candidate lists if your file uses different names.

## 1. Imports and settings

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

CSV_PATH = Path("Spotify_Youtube.csv")
CASSANDRA_HOSTS = ["127.0.0.1"]
CASSANDRA_PORT = 9042
KEYSPACE = "spotify"
MAX_ROWS_FOR_TESTING = None  # e.g. 1000 for a small test run

print("CSV path:", CSV_PATH.resolve())
print("Pandas:", pd.__version__)

CSV path: D:\BDA\Spotify_Youtube.csv
Pandas: 3.1.0


## 2. Load and inspect CSV

In [2]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"CSV not found: {CSV_PATH.resolve()}\n"
        "Put Spotify_Youtube.csv beside this notebook or change CSV_PATH."
    )

df = pd.read_csv(CSV_PATH, low_memory=False)
if MAX_ROWS_FOR_TESTING is not None:
    df = df.head(MAX_ROWS_FOR_TESTING).copy()

def normalize_column_name(name):
    return str(name).strip().lower().replace(" ", "_").replace("-", "_").replace("/", "_")

df.columns = [normalize_column_name(c) for c in df.columns]
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (20718, 28)
Columns: ['unnamed:_0', 'artist', 'url_spotify', 'track', 'album', 'album_type', 'uri', 'danceability', 'energy', 'key', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'url_youtube', 'title', 'channel', 'views', 'likes', 'comments', 'description', 'licensed', 'official_video', 'stream']


,unnamed:_0,artist,url_spotify,track,album,album_type,uri,danceability,energy,key,...,url_youtube,title,channel,views,likes,comments,description,licensed,official_video,stream
0,0,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Feel Good Inc.,Demon Days,album,spotify:track:0d28khcov6AiegSCpG5TuT,0.818,0.705,6.0,...,https://www.youtube.com/watch?v=HyHNuVaZJ-k,Gorillaz - Feel Good Inc. (Official Video),Gorillaz,693555221.0,6220896.0,169907.0,Official HD Video for Gorillaz' fantastic trac...,True,True,1.040235e+09
1,1,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Rhinestone Eyes,Plastic Beach,album,spotify:track:1foMv2HQwfQ2vntFf9HFeG,0.676,0.703,8.0,...,https://www.youtube.com/watch?v=yYDmaexVHic,Gorillaz - Rhinestone Eyes [Storyboard Film] (...,Gorillaz,72011645.0,1079128.0,31003.0,The official video for Gorillaz - Rhinestone E...,True,True,3.100837e+08
2,2,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,spotify:track:64dLd6rVqDLtkXFYrEUHIU,0.695,0.923,1.0,...,https://www.youtube.com/watch?v=qJa-VFwPpYA,Gorillaz - New Gold ft. Tame Impala & Bootie B...,Gorillaz,8435055.0,282142.0,7399.0,Gorillaz - New Gold ft. Tame Impala & Bootie B...,True,True,6.306347e+07
3,3,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,On Melancholy Hill,Plastic Beach,album,spotify:track:0q6LuUqGLUiCPP1cbdwFs3,0.689,0.739,2.0,...,https://www.youtube.com/watch?v=04mfKJWDSzI,Gorillaz - On Melancholy Hill (Official Video),Gorillaz,211754952.0,1788577.0,55229.0,Follow Gorillaz online:\nhttp://gorillaz.com \...,True,True,4.346636e+08
4,4,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Clint Eastwood,Gorillaz,album,spotify:track:7yMiX7n9SBvadzox8T5jzT,0.663,0.694,10.0,...,https://www.youtube.com/watch?v=1V_xRb0x9aw,Gorillaz - Clint Eastwood (Official Video),Gorillaz,618480958.0,6197318.0,155930.0,The official music video for Gorillaz - Clint ...,True,True,6.172597e+08


## 3. Detect fields and clean data

In [3]:
def first_existing(candidates, columns):
    for name in candidates:
        if name in columns:
            return name
    return None

artist_col = first_existing(["artist", "artist_name"], df.columns)
track_col = first_existing(["track", "track_name", "song", "song_name", "title"], df.columns)
album_col = first_existing(["album", "album_name"], df.columns)
stream_col = first_existing(["stream", "streams", "spotify_streams"], df.columns)
views_col = first_existing(["views", "youtube_views", "video_views"], df.columns)
likes_col = first_existing(["likes", "youtube_likes"], df.columns)
comments_col = first_existing(["comments", "youtube_comments"], df.columns)

detected = {
    "artist": artist_col, "track": track_col, "album": album_col,
    "stream": stream_col, "views": views_col, "likes": likes_col,
    "comments": comments_col
}
display(pd.DataFrame(detected.items(), columns=["target_field", "csv_column"]))

if artist_col is None:
    raise ValueError("Artist column not detected. Check df.columns above and update the candidates.")

for col in [artist_col, track_col, album_col]:
    if col is not None:
        df[col] = df[col].astype("string").str.strip().replace("", pd.NA)

metric_cols = [c for c in [stream_col, views_col, likes_col, comments_col] if c is not None]
for col in metric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df = df.dropna(subset=[artist_col]).copy()
df["_artist"] = df[artist_col].astype(str)
df["_track"] = df[track_col].fillna("Unknown track").astype(str) if track_col else ["row_" + str(i) for i in range(len(df))]
df["_album"] = df[album_col].fillna("Unknown album").astype(str) if album_col else "Unknown album"

print("Rows after cleaning:", len(df))
display(df.head())

,target_field,csv_column
0,artist,artist
1,track,track
2,album,album
3,stream,stream
4,views,views
5,likes,likes
6,comments,comments


Rows after cleaning: 20718


,unnamed:_0,artist,url_spotify,track,album,album_type,uri,danceability,energy,key,...,views,likes,comments,description,licensed,official_video,stream,_artist,_track,_album
0,0,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Feel Good Inc.,Demon Days,album,spotify:track:0d28khcov6AiegSCpG5TuT,0.818,0.705,6.0,...,693555221.0,6220896.0,169907.0,Official HD Video for Gorillaz' fantastic trac...,True,True,1.040235e+09,Gorillaz,Feel Good Inc.,Demon Days
1,1,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Rhinestone Eyes,Plastic Beach,album,spotify:track:1foMv2HQwfQ2vntFf9HFeG,0.676,0.703,8.0,...,72011645.0,1079128.0,31003.0,The official video for Gorillaz - Rhinestone E...,True,True,3.100837e+08,Gorillaz,Rhinestone Eyes,Plastic Beach
2,2,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,spotify:track:64dLd6rVqDLtkXFYrEUHIU,0.695,0.923,1.0,...,8435055.0,282142.0,7399.0,Gorillaz - New Gold ft. Tame Impala & Bootie B...,True,True,6.306347e+07,Gorillaz,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown)
3,3,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,On Melancholy Hill,Plastic Beach,album,spotify:track:0q6LuUqGLUiCPP1cbdwFs3,0.689,0.739,2.0,...,211754952.0,1788577.0,55229.0,Follow Gorillaz online:\nhttp://gorillaz.com \...,True,True,4.346636e+08,Gorillaz,On Melancholy Hill,Plastic Beach
4,4,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Clint Eastwood,Gorillaz,album,spotify:track:7yMiX7n9SBvadzox8T5jzT,0.663,0.694,10.0,...,618480958.0,6197318.0,155930.0,The official music video for Gorillaz - Clint ...,True,True,6.172597e+08,Gorillaz,Clint Eastwood,Gorillaz


## 4. Create processed outputs

In [4]:
def make_song_id(artist, track):
    raw = (str(artist).strip().lower() + "||" + str(track).strip().lower()).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

df["_song_id"] = [make_song_id(a, t) for a, t in zip(df["_artist"], df["_track"])]

def metric_series(col):
    if col is None:
        return pd.Series(np.zeros(len(df)), index=df.index, dtype="float64")
    return pd.to_numeric(df[col], errors="coerce").fillna(0)

song_catalog = pd.DataFrame({
    "song_id": df["_song_id"],
    "artist": df["_artist"],
    "track": df["_track"],
    "album": df["_album"],
    "stream": metric_series(stream_col),
    "views": metric_series(views_col),
    "likes": metric_series(likes_col),
    "comments": metric_series(comments_col),
}).drop_duplicates(subset=["song_id"]).copy()

agg = pd.DataFrame({"artist": df["_artist"]})
agg["total_streams"] = metric_series(stream_col).values
agg["total_views"] = metric_series(views_col).values
agg["total_likes"] = metric_series(likes_col).values
agg["total_comments"] = metric_series(comments_col).values
artist_summary = agg.groupby("artist", as_index=False)[
    ["total_streams", "total_views", "total_likes", "total_comments"]
].sum()

for col in ["views", "likes", "comments"]:
    song_catalog[col] = pd.to_numeric(song_catalog[col], errors="coerce").fillna(0).round().astype("int64")
for col in ["total_views", "total_likes", "total_comments"]:
    artist_summary[col] = artist_summary[col].round().astype("int64")

print("Song rows:", len(song_catalog))
print("Artist summary rows:", len(artist_summary))
display(song_catalog.head())
display(artist_summary.head())

Song rows: 20626
Artist summary rows: 2079


,song_id,artist,track,album,stream,views,likes,comments
0,ae3b6a2e67c84d9f2068672f877caaeb494e91975f6f9f...,Gorillaz,Feel Good Inc.,Demon Days,1.040235e+09,693555221,6220896,169907
1,3ee5e9f049e46cc310a844c439948a5c817c23aefacbd7...,Gorillaz,Rhinestone Eyes,Plastic Beach,3.100837e+08,72011645,1079128,31003
2,f2584155072d63cdbb348df890890888573fb5cb565cb2...,Gorillaz,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),6.306347e+07,8435055,282142,7399
3,034a07c31ddecd04afa97335f7e0bbb206dc79cf96f69b...,Gorillaz,On Melancholy Hill,Plastic Beach,4.346636e+08,211754952,1788577,55229
4,459372a89efca8f64fd7998ec1dc46a14deb6fb1fdb21e...,Gorillaz,Clint Eastwood,Gorillaz,6.172597e+08,618480958,6197318,155930


,artist,total_streams,total_views,total_likes,total_comments
0,$NOT,1.127638e+09,110784903,2747163,81497
1,$uicideboy$,2.126748e+09,334135108,3576621,65127
2,(G)I-DLE,1.304212e+09,1754953941,24462963,1456246
3,*NSYNC,1.427834e+09,1027832862,4233423,225267
4,070 Shake,4.647486e+08,96099359,2013005,41366


## 5. Check Cassandra driver

In [5]:
try:
    import cassandra
    from cassandra.cluster import Cluster
    print("Cassandra driver version:", cassandra.__version__)
except ImportError as exc:
    raise ImportError(
        "Install driver in this notebook's Python environment: "
        "py -3.11 -m pip install cassandra-driver"
    ) from exc

Cassandra driver version: 3.30.1


## 6. Create keyspace and tables

On your current single-node cluster, replication factor `1` is appropriate. This cell does not drop any existing keyspace or table.

In [6]:
cluster = Cluster(CASSANDRA_HOSTS, port=CASSANDRA_PORT)
session = None
try:
    session = cluster.connect()
    session.execute(f"""
        CREATE KEYSPACE IF NOT EXISTS {KEYSPACE}
        WITH replication = {{
            'class': 'SimpleStrategy',
            'replication_factor': 1
        }}
    """)
    session.set_keyspace(KEYSPACE)

    session.execute("""
        CREATE TABLE IF NOT EXISTS song_catalog (
            song_id text PRIMARY KEY,
            artist text,
            track text,
            album text,
            stream double,
            views bigint,
            likes bigint,
            comments bigint
        )
    """)
    session.execute("""
        CREATE TABLE IF NOT EXISTS artist_summary (
            artist text PRIMARY KEY,
            total_streams double,
            total_views bigint,
            total_likes bigint,
            total_comments bigint
        )
    """)
    print(f"Keyspace '{KEYSPACE}' and tables are ready.")
finally:
    if session is not None:
        session.shutdown()
    cluster.shutdown()

Keyspace 'spotify' and tables are ready.


## 7. Insert processed data into Cassandra

In [7]:
cluster = Cluster(CASSANDRA_HOSTS, port=CASSANDRA_PORT)
session = None
try:
    session = cluster.connect(KEYSPACE)
    insert_song = session.prepare("""
        INSERT INTO song_catalog
        (song_id, artist, track, album, stream, views, likes, comments)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """)
    insert_artist = session.prepare("""
        INSERT INTO artist_summary
        (artist, total_streams, total_views, total_likes, total_comments)
        VALUES (?, ?, ?, ?, ?)
    """)

    def as_float(value):
        return float(value) if pd.notna(value) else 0.0
    def as_int(value):
        return int(value) if pd.notna(value) else 0

    song_count = 0
    for row in song_catalog.itertuples(index=False):
        session.execute(insert_song, (
            str(row.song_id), str(row.artist), str(row.track), str(row.album),
            as_float(row.stream), as_int(row.views), as_int(row.likes), as_int(row.comments)
        ))
        song_count += 1

    artist_count = 0
    for row in artist_summary.itertuples(index=False):
        session.execute(insert_artist, (
            str(row.artist), as_float(row.total_streams), as_int(row.total_views),
            as_int(row.total_likes), as_int(row.total_comments)
        ))
        artist_count += 1

    print("Song rows written:", song_count)
    print("Artist summary rows written:", artist_count)
finally:
    if session is not None:
        session.shutdown()
    cluster.shutdown()

Song rows written: 20626
Artist summary rows written: 2079


## 8. Verify rows stored in Cassandra

In [8]:
cluster = Cluster(CASSANDRA_HOSTS, port=CASSANDRA_PORT)
session = None
try:
    session = cluster.connect(KEYSPACE)
    print("Sample song_catalog rows:")
    for row in session.execute("SELECT song_id, artist, track, stream, views FROM song_catalog LIMIT 10"):
        print(row)

    print("\nSample artist_summary rows:")
    for row in session.execute("SELECT artist, total_streams, total_views, total_likes FROM artist_summary LIMIT 10"):
        print(row)
finally:
    if session is not None:
        session.shutdown()
    cluster.shutdown()

Sample song_catalog rows:
Row(song_id='64585bc9a63a833a4ae4f16d82e1c08b0f0b02081b38439fbf51a2a90e6f7d73', artist='Iggy Pop', track='The Passenger', stream=323795115.0, views=31172789)
Row(song_id='10f5531432b363d48718fc93fc78e89bc216419f3e2ebc78e9e01178b719251a', artist='George Strait', track='Carrying Your Love With Me', stream=111639862.0, views=52540392)
Row(song_id='d6e297ca0fb0581e0db01650f58415b366dc2cd98ea86c2f2f9174081dcec351', artist='Jaymes Young', track='Infinity', stream=782970821.0, views=47368786)
Row(song_id='4d6eda2a1825a3c630e4b396bf3f6d0420628a9cc6663fa7bfe5b5ffe0f46b00', artist='Miranda Lambert', track='Bluebird', stream=136497255.0, views=23238672)
Row(song_id='76b40e8fd0eb167885c1364276a7524609a3028354fd71d44159edc304829e3c', artist='Bob Sinclar', track='We Could Be Dancing', stream=18778096.0, views=8127251)
Row(song_id='55fcefb52fbe230b62742f9138ad7848f740472df64245f8d5ac9cc1117c202d', artist='Swedish House Mafia', track='Lifetime (with Ty Dolla $ign & 070 Shake)

## 9. Optional: export processed tables as CSV

In [9]:
song_catalog.to_csv("processed_song_catalog.csv", index=False)
artist_summary.to_csv("processed_artist_summary.csv", index=False)
print("Saved processed_song_catalog.csv and processed_artist_summary.csv")

Saved processed_song_catalog.csv and processed_artist_summary.csv


## Troubleshooting
- **CSV not found:** change `CSV_PATH` to the full file path, e.g. `Path(r"C:\Users\YourName\Downloads\Spotify_Youtube.csv")`.
- **Connection refused/timeout:** ensure Cassandra is running and listening on `127.0.0.1:9042`.
- **Driver import error:** install `cassandra-driver` into the same Python environment used by Jupyter.
- **Column not detected:** inspect the column list in section 2 and edit the candidate names in section 3.
- **Existing table has a different schema:** `CREATE TABLE IF NOT EXISTS` will not alter it. Inspect/migrate deliberately; do not drop data unless you intend to.
- This is a general starter notebook, not a byte-for-byte merge with your existing project. If your existing code uses PySpark, replace section 4 with your actual transformed Spark output and keep the Cassandra table/write sections adapted to its columns.